# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz27aW3/UrsOx0SrYyKzmPptqqs0nstJl1Em+Stnev6+FAIigx5qt8yFYy2b91v99ftuccACT4kp3c3Exr8QEcAAfnjQPQsqwXaZnmCxbAj1047OhnFvMo4uzeCbN/rZJ9xZN7J0fH7Ht2HO/ofSluyirnePkkEokvcncy+a1ImS9YLtZVXqTFYsLYt+xt6qcFi+D/OPUF/v5VCZ8nbM3zDcd3ImHQ1i/nvzGOFzvxntmZyIMwYi8e/+35i1fse4DE2OOzt6+cKeNFyJKUbfmeiQLKcYRQ5oJVCachsD2CKcJNFcIL4VI3zqChTECvWCFYBl3E1hPV9uvHL5i95uutwP5nfBMmABUuz8KkunFYJPYwxpR6gSPg+Xob7mTXM57zCMa1YApoDh1RY2Ulz33oithUUL2QPTmNWJnzFX8Hr0WcpVCd7TiC+t4Xu++LbcxsPyzWKT6CnjlqABJF1Pu0FBvB1mmCTa154gMe/1NPBLOfQrnvqCqUyQEzIqGu665STRNFbCMSQOQ6TJNmNhq85aKoopLmCprfVDgoKogYl2iTXWUwGpiYYou4i/RsNGh7VxWlKBBTmSjDHIvEYRGrgqJgYVKUPIH/ROpOLMuahIigEjsMNJVDX92gAsKDojBB60C/36zrqyhd6WvsSRTWt2Gqr94VaaKv00Jf5UJfFduqDKP6rloByteiqEuW21xwP0w29YMwriu/DzMgXTHR90kVZ3vsbpJNgjyN2fnzM6bePY/5Rkzlz6usmMgCLmuGHYSbyeTX09enbAlddTNebl0/zBMeC1vf81WBv7bnYcOe5zgTmD/v7NXT/4ZadWfds3R9ZTuT35+f/vEGXtgWtJaU1pRZK76+wt9IBHSfh5ttaTmTyeQr9gaoNw+BxBbQLY6UUrBNCMwDU8lSFvCowMGlLAO+Bjooq3gFTCAiIJZ8F/ppPqHxucDM3vMXj3859c6f/+30DLvw8L43m83w/wm+ffP8BMd5PLv/CJr2RcAKHgiPmrXDeOMsiJaAMH7nEXQJOD1Jc5BV4XtOxCg7iDOGb+2SxzxJp8iN8DfAomXquEhYCCcMoDrQRSHJbk1tqNlw6a9qEP/lPATih3YrcZrnaW5bwMhKEODoARlGD3bUP8AgVr2esi0MC4C7Rfhe6Lav2U/s3jFDlqGrQ22d1ZAFCqYYhCDwI0g0EKbQqh2HSRinCC67cVSz1MS3APxnNjwBn9HgJgdpI5oGYn5j4/AcaETPYAMWRqzGvU6zve2YL9xyC4SS8DCybV1zWsNw9DScPX759O+v3tQNYk0UrnLmEiDi1788QZKFn8eWM9x2shN5aasiVCIXIEUSfK0IbR0JnlSZl1ZlVpWFfSVEtnw0A9oJEy/IhfA2q+V81lDgE5CsnMSaIR4BTWwXincoKUNSUTXqFK+UnJFwr4mwzPdNp/0C+lwA6wvf9pFgmY+DRKHm4p+a5d+lYWJL8eC++u2td/L8NeDgW8txEEm6VFiAqLB9p0E8jgSbIAEHgqS48qoCuasNy3GpIOj7b7+9N6trX29BvmAngQiYDerG9guceUQW0jFV+slEmTEhND5o24fOpZk9c1pvVI/yGDS5sP0pCzfA2sITSI7F8m1eiXYFY5jiJixgynxQepYL4tfqtIr/oGwu4nQnzGKtUp+PGnGzFlnJTukHdGjTesZBaUwm6wh+2WkCdoWQL5HkPA+YtvQ8uxBRMEW1HixfpgnogyjdLLM8TEpjIFjIzaCDWA6Rrbrli1KsS4O3qCBAgKLwt/0YULDCF9hM+02x5ZkYf+PFuwMvy4MVR15nHAY4/sa7ChMkF8Um9ct36QqZ5IPF1yXaYNaCAZdaIvHWacTx7mO7ggfWHvDkNVR6BrqqaesrUNtogAHbCj9EDrXxAmySdBdGDplLIP1At4M6W4MNXCoObsMXwNLYH1Iv0P4jF/ujDGS4v/eAHqykdQYPHs7cXh8BiIe0bGj5YRZXXXbRhDEIuCVFzK65VebzUthYHqiC+3aaAd+2W3UMCTFOzDVBt9qQ9h8g4GOX1Li/7+GcXilUeFmaRlBiHbhvyUY5h/vTG/AewBuxQbN412l+JYD7j0GNScq4sNQz67LpMqAj5lcwdXnRQxjJBi+9UhKkYT6RFGBHEiIMLgOp/CZE22Wd+uEmRXsX6KxCC0OZzYYjs+e1EzOFl3EWIS3AmxocL/NwBcMB+6gS6DAQTSVo+YPBjZ4QaA+QAuuQkzVciAJQXusGLTLlbHl+uAZx0bxxC1HCWDhoH9tSnAiqEBnKOVgKefKWgg0PonK1xoohN0KBQ9w4VlXzJVQnGhkrBzilFj6Nv0Zb1TQPMD+F08bgEflj/4wWQTX57kbAW5PSLVDNBeG8oxMv2sUu/xWWmEjBdvQv/JM0XTMKzBKKDe9aoE9QKGUFHlxaLG2YBjCbk/J7I0pg0ez0Hsc7QGKL0c5AjZZ1dKD2qdEDJ0edeMUX73ge1VECcDJb3EH+0rbagGbdBHwtvG210t5TAZ6xR/MDnJM3VcCGQQ1ycTkuPdHyytHyasOwHZcG3jcvYMpzeueFZLGNFKthix2VcpEFkOVHyjb9/Q46XJS5Hbjk45HUJlCBbG5Hz4vLUTmOHqgY0BFgI9jWS5J0WQUWfiQEiaI6HPLrswVMacf6UvbzbNKwBdBK7tvUsXYzCSB71npyHYKmIz2ExdHfXEETqyoIBFg9m+XMwd4GfZxIAxRF+TC+VtCUVD32w/vsp5/AlXQGCyrXbzWOdjXCZLBAgvOB9u/KaTCAxgwGAlz8YxhkNNphjp4/pJGKm3Y/yrTkqBmLKrbFDSi3zM7BJKYZdvqWnh1Y55JjVAhKRdWAWRbsgwSm7NWFOw8+sl+e4GurP7TA+mCMgB2xcuHOgo+FIf8VXgiqoU6lYsnCTBgSYoqe+JRBhVVqEIVk2v09H9xLaSWin6m4thYZJ+HbZ1F6/YKXIBiSzTmAjkLDVMS2AEe3lXexNdQ1JTi5CfhVHTpWvQzSCCTwkvpbFcLDwANIsaLvfADp0IDaU/YVewZG/vb3E0C5L9B6CMI1xTNlKPX/5rMbcg9znoUqlDgqfvTwXPDtQTd4AYLe+bEdrz0ebdKlFa873oti97f7THrwfYCDrYy21GebOwiUvmBhNVrQ6gGnKkuTEBoaECl3hP/JsBXB4igbeiVLWBLqNRj2YBHRLKs7Mn7MBzTh5oO/0CQs9+SvtbXaU2RA00x02VuszewHaD3D8+/qfjsYXAFjE+NTFDuFroEzZ5AWBVHB7mEFjyjKWseS2VNAEUaEZUiGlTxehVCAouvaLgXgNSgVkpYBM+jNTHZH+zmqF08pcOW3lKwOcqagohtph3cuBg7BdincdeVzEFRlXEUuEHx67ZXBvWPgT0TagUpJ8qnFV2BbbMElu+qWbokV6emqnj8BCJs8BcS9phhAbjKzOZ/DFlqrxLI2u6DfXpp4m6wCmw0mDxVKQy236wVo23DKB1tuOe29UbRhNU1TeKblno8DJ04imgWqYuZij6JaSX4ktbjjum5H7gxKlU5oQLovhoYYNhtHVfHYP2urqvtHflge7Y6PZkeEABAA/YDRZ8gXiYLbBNcAibTosktj7dn5rLmRswJOp+TWgXlpBXXuOAFDCO17Z3o48Y5GMk52FPn5jNEBYEV7O56HFFEtSl58BvGp2NOdhg/+yadT4CARxrsvS4UNQg5RYmda7BbJgXQamRinP2ufOWft2TpEm7dP27930oamrE/n/8JsteZpiq5VVaCSLVJQwHNa7RyZPRkUradE3g7OR0vbleJmyIQ+x+o9w5l6XFEU1b4wcEGNmVL0sqNWBOAH3GYHLJjhetZlz4su0B+nFbniAN6CIaKi3jP7wzfU+jfYmfqSIMoOfSMtmm8+DlDaQbt39EUvRD2M0MOeRY/ohkVuy/UY9lVvN+0NKrLk1CkMLQ52iVb9vigGBgfpTA6D15F+6O+wUw9u89WXckfuoswPL1GYplvNn2j79Xm0ie8XApiIQpbgWobvpZOMRiQ8NEPZvQYurDQI0FuxLhe3dFT7cGToe+us8lTVjjOHXHOL3aOye54/buOL2WuQYDkuQe3BT0EHo0n0cKyWOEOMDFnK/UBbRuu4Lfk/7aor84EzGH7LzGkgCVo/Mc3RERfYLVPbApRZzvAqRsugozfXPI+rzB6MxrxQCwQRqEIZirk1rKJre7FA57t2VFUz/WWK3xKODhz3MYcmCcFGB69OJiAtaDEijKQbeiXyBKYcZ+rpbyeP2R4IINw0o6Hga7RJ87CMZfh1XZ28fCkTrQBGlkOvc53MwzFXiWPcFjO1onR9N29RBWtajkHLKDFJhworpacdK+kKkpcZFh7fcRgeUKTdIQbp7N/qevUDvkOJOSc5v26zfQzwZF5CIq5VQsEUfOn58ZTBHweuZ1Om/nOc3tp3DdbFP3YYd4q4IorCrBC2Pf/h4ZTdBzD37sHF8WzmUPwvWtpwPWXzH/DPA/hz/OBBtyGgWmDOZBMhnB+h+vxHBDRHiD80gB7CQ8xwmD8cBIMup8T6L5QdhvFKX+zCtVhaOA+WA/5+AgYmCDcQ/Z1AKwU9ZfUwCUSOeoEkkw3ty+e8KtM1L3ARBcFNmV+CilvKlyi4yvlDZzgwZlNuyjKMp5he5TUtFKXIiiXMRrrGRAIvF2BzVagflvPjRyO2IoJYgz1zVSy1xJK31uVUpcZBp5YbgErZIR710yqRMYpt17htCLXYJ+ttDqLzvbCdyYjhgyEcMJVBaSIr30VUfFJkn2ngMWb6pSyNwzL0tflpxG+HFBRIXbg0k2qCW+3SLgcOclrbbnX9MKAJLENSYDmthoncRfSqO82YL+DRa3oyGdGCaiHPHAxKHp14NV5DNlRXUJQgn3YsW6Oy6t7S6JmtVgdN6uuC+yS3xRzLkqak3xucve44P3cNaI2xRJnDJ1PoUCuoFdcu5Wid1dFQvXlX85cVlFo58KatMPDfDsWtrODuwhyX4bGpNN/bY6JKhiGpjIdxxTXHVCpHL39Q8H5Mk0h3YjbGqJjfWpG+/bCL3aoQfmtR5Xt82llrmcm1ln/+Y2itRa63IPt/2CgIiki+2eU89jarby6xuvVJuRlfYgFYp3maa8CZEgiUISkzqbzVRi6NtOQD5t0dEAxtWxrLd7IvizJ3upSzqdUuLRpiJmgnz6+TJzoZqPsqKzBTLPDAWUnAoi06RbccqCHKtlylDVKW4XKpMgvJokwyl+c531PFC3A3QateumB+AfH8BCp0Zg6sRpKsmpZNC0Pja0K9djdp0cw+OJzWKCcf1D8wFPAv0GK6eicoJR7dshx5WiaNV9zP0X7HnJYYrEbRiGxEwOBI6xL7YspuClksSZP3Ik9tzn5m81nLBcAF0puiM5s3YGzczKdsD7/7OcC4KSQGp3TFb/BqXz/bq2cDGCMk5Glm2whzT3DZd2xOcOHXwNpQBi6xNz5MSrtJY/2Wzd35I8Mu/2vY5AMMFKPmXvEXKAcwRSRF23YBevwahBAIBqwKd1t55/SWqaAqaA1UwPZ8dnwfrDP428uGJUVMCD++L8VW8VcjkossCkuv2ApRajWO12C+5BhokGnfU8z1nuLaylRle7fci5NwF/pyT8U2fYd5GCqwJ/c8gEchcDcArh3hajMuXgVp4qdtVyCQLR8UCEbiWL5Z8TYrjHFuJ3eU2uhlc6tRo0SRSDJkCL1znFE+azEB9qvmAkJ7XW5NKWzc5cneNIELsQEKgZ4AErEHPC8p5URFGaeGmkEXOJwySggRYIqitYlJp1FXDAZsV4sSCXnAEe60KJcsw47/r1AlwR0G1cvgM0fo8gxw6QOBY3PQjtPi/yHIQ9VQTuB4jdp/EFLJM7yYX7bQingsZGyRolz4CAkNygFjFRcznKE/gJFn7uz+ZVceYXES1gsSlUAsbFuBvCwWuA6KWxrCDdgZouilsGBtYiCX+Mu2ppbj9MaGvbNRpITQgz/AFEicKYkYeIAyqX4qs3dCyt4BX03YidN0Frd3EKjUBeoNM1uWTrF0tw9NLfBO2omYWAdBgdCRYtdBAO8BHjUwpR53yAyAXODby9oGRuWPHGAIWyXD6SkKK8C94/Qi6IrBAeIXMUtozavxWKBbPMyVgENPYYq7ETzMAivummxinwjl34lnUE+tq+JA8Mav1nBzR0P9GTqsIlcgGiRgz5CJWq9tx8bnvVKD3ekVHrWt91gOrNnWyxgJqX7lopvyRpQdjRoDU/q+h4Xsdllq20UZCZ5/ITHtSiT3QOC7MNl4yvTxKyBStMA9Xd0+UAVn0wNrwgOHI8SYuCdgzg5UWUegCbxtCsyKCpwu0EE6ns1mB9rx0hxdYUko3jrdkucZdX0K9D1izDmgbWcSM86g3yLDAO3KcjrVK/et/LVjwqO4gSHCK7J0YsKl8UBtNVt2eOkTnf56DofXonTPBgLgg6siB4aqhwsYBitybKYlZIkNOdVuEN54cumm6ND3aIUo0tPdKz+In+PDYe6czBfa9SjDItK5Pe4ZsQbNs58bKbMYmnNDdCjGbeQSaPdlI6O6cpJQ2xhxFCVF6aVEHOZKo8jGCJeOb+F8Cn/KNhV0HkwelWvXc8uGPOy4isoQ9cXVAiwBVBVXygqhltywFDFMo7Q7jNh6o170erPUjQQP8TNvjCOwL+Qy44FwPjy4rJPq1Brj4cWAS4f92TWNdF/UGmWnC5Q9a8SeTZAHGm+1XF52o9mJuCntrFnDkG111yKaeca8QkKXDHDfuuhh0GFBiQxD9cpD1QKq2RETSEAo3MDBQsOE7kHwPHKkzaafsJ+WYOhKnDz4vJWdtNyKvNPhNk47ne+PZXCtR8HtrPe0niKqhtd6qNjAek+dCkmvqnr35kBgV8RZuZdp4XbXNZdsBZ1Us034I466UPttGzr6xBA7zQ1cOJ3c4i8eZleirBVp3wyH2g0KGjWU+rF4rCWfOtOD4fdRmFrieQXGt5c0Dls/dNph+/GODYbzwYMYEs1TNaNfxJZVwdRa2qsIb8uaVQIf7YXPDcMrSG0PLgxqmOw/loCpkpelNqQtjCPrvUED9kV/oaDun8kI5Ly0qf4S00bko6tLqW/kBt6RzeekeEgT4VaaK+fAdo2v2O8y9ygXkjrx2g5yOtvgOzwmgkfAtq7rMLmmCVYK7rpgUcq4D6pfdF2g2vnhFB4iLNacUGi7AK9J40kux1sgns822vShGhiTayVToRFH/QTynsosIjm4Afvt1s63eti4aDJZ7csQd6oS3JGyr8S+bcCkWWn6Zyho1EENKPfnhkCtaYROLGjjDdWoSR39UMnBrIyt3hd55QK5kLx0xgvtQCes9iVatReLBfvxBwOD29YOS7+Ks8LGIU5pH7cHw1cbCAYaUtjfultx44fg7JTYwPz4sjED8ypR6JNRO4bcorcJA8OoS+IZdU2hLXndD5N5B2OA3dD+tG85qEak9FY3ZM+qayBi3Q9QVcv58b37bShaSC9p3x60u7S2FS4yhT5upNvSPrPl3H0EL1MQTWI5g4LtyYei6dJ6ysGG4bk1lQ9QweimOW7GrLsESC5xjWdgTEqUqdz+XESh2GGLD01TmryyTY5uWcTjlc8ZCukFU6hIPAwdhTwaQDr1THlzVBT4eSO6+wbUdly5JXU4jYYyUQZlRyvB57xm/knX4KtVNeZUnF9YdN3aR0tzqkqqG1lU3phlacpVUXktS9K1WVAuS2IxvJKFOklP9eypgs29LF7fm5XUFOtxqTs1MnnX6saskwrSGPNqam0gs2NQP/9ThSVlIlI82zK7CeJGbvUmjbYw4mOKOWvucfShKmYhYjSzTN8wkUrQrEQs2gYsedUsRE/MUh8HY/DD8tB0/aT5ba4fqBh6Izsc7RkOmuamTB62TFroRFMABPmkC6G2Gkasm955Jc/oaA195pAvlII0pg+FMh4/Ea4VGyxbrvRSe9RSmNFfKc3gfyXMagN82KSsJVvjjqN4g/9rsSZ/askmf6Yj+EEhR39NAVdfNVJO/Q6DaYTfKk0ju751prXsky/kjdMIQWlTqztnFDrISA15lTpTLfHombw2NN6VFk9oFpgGgUkBVzUlyk2o5Ny1jjRpXgIBXVjwCHh9zLB6rc9nwaUrqoRm09XghtYW4EZGdE+HcTox94M7yUnqFGUekOSxvv7fr+Ovfe/rX79+8fUbz8Js6qvhswzIKuscYNA5L4XO1Wm1jmdF4Q78oqz8MHW3ZRxZDmUuGZVRB2G3lTpCsWkrzSbXC/xFJ3aBe8y7Az2XW889Qv90bBRYZHQYtEIxFHZafKJBt3MLvhNtVMiWA4uMaO9D8tHNkg0u2tS1TzU5CuNUj6/YawwMkhiJ2Hw2Y1/LbXfoLuCRDXiqGybJoB3AC6EPrcMTsfIwzY3ta0gbpxfq6AJ0gOzTi/rogkvliSHbqdwTJIbT+jwAqnG/wZWn9/Pgyha8mrk/zvBAuaaB77FNo4JUvrpiXePWXkg4rb3dZbi+ssH7WKEsXONCQEHZGmnWmSvLsh7vePKepPEKF1BZUcHcxOSN+ZiGQAKajuuT6Ms3vLVuTMibj+RuNvvAyTaCDmAeT4FLGn2yuFGBLhj4gymzO6l1cxwprv3LEc0HfIDaOuA4eyuoxXEN74bG3y9O3bnmYQktPnDaGMzAzRF9FAZJV3IBjNa5bac7QFoHA3RUTFNE7i23S8zkKJc4V1MG18VyZMqAwzkYC8nAwUrAPbQua99hXUBJzSDpFA7ALo+ioV0+iJ+it/yk2iWuNbBWH/hzUZ8ocomx3LmpLfRRdyDs11eY7NXJijLMu3sgGE8TXLPnC33mJPAsehEsRbKUe2StTnxNtzCgYLqdOzI71ymlj0jpDGEUuclOHQHQEn0kMg15uUMjqBhyX8ncGeMgHeajqJavwtJEoIAnYBUlM6aGdLnrSqglg5loQtfbl++dWMg/VndPn0U+zq6JoVgj20X6/wjcP/8ht8ZZnSA+gLlrd6XeW5iRZ1p1ObDe8jn7BAfsRH0ywsDU1aK+I7GKcQFVz5h1FsZZSPjHtK9cTgXfVO9EnhbWMB3UO+7Usrqkjc5CVWu9T9zgotKQxrV4vg6jiLubaGUNjG7Udhmq3jFdWiaMbdHhCpYsY8YDQZ9nnM0XzQEMezxdCfT0p7NJox9H9pHJlz704kppXMlJpt6dzadSHxNH1cr3E0nJOg+TUnOWGS58/vhTN0K2yF6Hn7uB54HJG949NIKHofBzQ+OyzK0UruEeojgpuFVgOnwPxv7nEt8YpMN0qCe0TYmaDo8XNEm0eeru+xNH86LvvCJ1t7zhVv7w3bT4kFJrqT7DJkVz9UHbTP1OPVJibtLnNOnzERm1m29Zrgbo5mEDHJ5NDKten48cVIIOlor00cno+5AtADJCrhu1T0em3W2uAemEztrBPA3cO6Xs/wVYtKB9iEZ5fZzVFJPoybaQmf94Uq30ihuA5WrshIgk6/pct1Cp0cnz5jwRuWQgj/jAM0BC9FTkzC0oBU72iCWh6iVtFtuvzTQ4InRa1rHkGHGphupJF0KBkEvousjl6FGMZnigibAVGA6RITY8O44OdgOjFa4tbM4H+wevZbP6CFjZsybiJXpQPrYNcYE601YmfJu8ZF2gVKIudA2MJ1N240xGdC/KMrBIwM/4AaQY3aOf0KpNHkOrJ7vxnux6Pdkd6Il+h8ii8sXGJP0z6bgCfZMfLlAVggQMOTmtxQJpQ0UM5DHOjK/yMG9IxoAlEn0YpjadI67JHnNp3C8To6jPNrNb9sAB2gf8l1UWif4+7G4IQIYd6pWlrv+Vr4e5LrjtBFqoOeCDmmiAEt1x1puNzFMA3aJaxeBF9n0JVQqTSrOpXLgzAow6sqjCcEbgsA4XdtY/dDywjvvhxGgGJ05pV9B2g600nyiAsiJRpkr3meshUorXtxjQMpN8g96ZsriBXTF3zdjdpV48kV4Jm5evKMzL6WhA+gjABi0aENwiMQ5LYlmEe5j2rGOFfAVGMVZJcKPE35+fE137NV0blFz39m4T9e+arNEJkzibSnnSPdhzJQ8DDoQrT6o298F6q6hvgq1MNd7EogxtawaotLZdReML9PXaqH06ZQMH0MIQri3n0zc6SSUZpO2057CnNQnzuG6C+Stp9/DcPiOH3aCWaqRzii4Uc1r90GvBlAREMmep3GsdbedeyJfSsJnWh3l5ZBgt6Wwnu9R+3HzEha1rKYh1PYIJtZoSap7qEqvoLmBxE5uu0qaMmYTfj/oPQlTfe1jK5XD5SY/l+YUlr2hzbVbhA0ofazDZUAviVBFMmEkiQV7CICJu8pULph4v1mGol1jV3ssl7vow1n4web9PEQo/uE6AkBG+e52HMIPINM7QGuGPjzCIFGccxJ50xUByGMT0noivKutTxjsmEGbfU+PvswH4czJ5zvDcgNaZi/XKlR/mS1k9zJbvsymR3VLi6Ursl1daisjvZCyDnX5AFsJSGgodMwQM/bDYelKq2lCzQ/2DLspOiZWq7MuVhhN3h1lRttllSIOlbmG6Qyy8c0b8qt0Q0J0zclJoErQO/NWGvTzzVzN8uJMH/rbf3hEiCAqqcXHpQO9L3Boz6stzj4wX+AEFWbdqQEBUwFtlG2NbvSKj3bqwaiEgaYGkvm5FvxuJ0DUn7CMye0VaTP0leNpgjgsLaUl2Ngn6h7PVx83eQQRwSx4+ux05vXerBIT1Z/JngkHNXUtS3IHZP+tgKTkfA/leo066dgnIabrsnmHSMU7VdxeknTv06QVTWQYH1uKCHW0kwUa9NXQMcx7slnwxtHwvlHTX3rTwo5drYYi5KMZOVOksWWBJevZfiKZwHYtym/rN7iJz5oxd7TFKdxDxKNzhxmovFKhP/7h/D7Nn8GtDCSLwafPm+bl3cvrs7PHb0xN5ZjyK/QisuGg5J8J730/OIDdIfRUEKBhPlcFFcLKm+/OOQLGTRotv3r56fXpCU+fiSZrYWdu23Di7jw4V+UT4qxwsabbit0T+HD6QCn38oRH1Sr9XzNLnuwCPIWkwIHNw8a61iJwLMN3XCpHIQyo98dewpKNqIjx+AIMVOrZz9DPjO4ytoyvuAkbxOBtcqFXhGqr/4vnZKaDk5Sl+guhDs+tVuT0LdtEElmkxxFLgF7QTntdrG7LQsPXT9oxcPCGmAUOWEUEhx1uXG4MEnb/Z1wejE7hHJjhgtHUVETxZYhQQRaxyvW0XId1rj48KyHOLRqHEPJOfR1MOCILBo2oaMPrFKGpkuMDXK24exxOMfMLUA+zQqSyAI/rl7AnTr+8M71qsFDW7eLqNhUDw1KM8lcfdwvu7AwNjxQ1WNwSs07lnT/52BzhZKN5zwvYPSAbndCs/llYBwu8CQU/8DzjxLzCoXJPDnccRpfgxB/fRQwDxu8yAFoVEC+ZLn706ce4MTMkI8AfIYn2sPoTVsKLlGIsbykEd4Kzf8UWLr0SxBpd+uCODFHxswKl5SdLwGBQV5SSywpgVMQKO47UO0mJPqNM4pKbcYXibEOjr3sMZTVQPXP12BIjsscbo7zLEoVlRIfNjkxmsDBd9LsfnBhiMgMKAaWWkw8rEVdzxY8kYj06JrbNgKUW1m9q6jvHgqwuVFPXk7PTlySl+XuOIVM3RUcDXZZrvjyjnoMroIcWWZ3R53l/XGgol6pAMsmq2J0Pu6Eg2EOLsEnrgBjAk41Q9qPASKlsKf3AnUWjR+QG2vJFgJU7VC3njDIGrgzIS9/oJJbLKyvW9BKzTVuVLdTcIup4zHOJcL37LR2oBXKLvqBHR1NuxZD5sHefXUvNMdZsPtKg2ZLLYfyxrVVmf4VZH9nXjTeSf9h9ToUbvkkOhGuq8atYOGksAz5rsfLIjy+WJkWq7rHtO9j0QGw7Ahwlemi+fn5/SczAizedv3p68+u3tbQunGGZRaeCrKqAtxnPjczCYk7HsnOeAHUajLctd2ZteOpv1mM5AYEHOY0HRTqqj6soMHGmnWc/w0KUBQy8mgxdcSXQ67bwFkdl/+t85MHMIbdBdjuvgrOJjeeqszAxc1hJ72NVrauGXB21KrnLvgVVMG05cPKU8s+cOplfdp6PrBgTrgBsJQqIMk46XQpOvj4wg5OTqTIR+OF7JIHkC8rhVYKJ7XN3rUkNfk1MN2e2kg2YRZc9vWUfpQho5ZbRHDBcopJzu3PV7qE4Vsa/Efqqz1donjMTFWHwE/A3281JSNjYEMDQuxg+UVXxAp1yMb7gzCKebQge8QolzBtunm/IGUyjB17akuCdiaEVuM/yIDgo/3FyDsmk2kgcsoV0c3ZvNZov+Fij5ejKZeKcvf3n+8lRPiPw8JMgrT9D3++zux/nwu4w8Yqqa/k6lhtJLX2/Anzbw6hCC6owG9v+FsGBO", "blender_rig.py": "...", "app.py": "eNrVfduOG8mS2Ht/RbqEsyz2FKtJdrekpkQtWpeRZIzOtCXNOYY5jdpkVZIsdbGqVJe+iGjAL7vAArs4gF8WWNjwLmDYD4sF1vCDDezbzJ/MD9if4IjIzLqxyGafmVnDGkx3XbIiMyPjnpHRhmG8DTORzPgXdiWmzHydcM+PuswTARNplsPNCG7CLInoWRhlYhpFFyxibrSMWMznfsj34nwa+C6HZyH+nyU8FSFnnMHVpUjhU5aHLMtDAAHXL4Io92YBT4RtGMaev4yjJGPzIJrq609pFOrrKNVXCQ+9aFncCX2VCjcRWdEszadxErkiLZ5ki0TAxMJ58cBfir2iY5o04ylc7e3NkmjJbKbewWxm/lw9FCHMVuhXc5E58sne3ulv3777wMZsYrzgS8BIYljMeBEliaCrDzzIPa4vM3n1nPuBvHov4iiNjPO9s/evPrz6iIBWewz+PQCEL6eJGDEz5mmUWixys0QIi7kccGyxTFxnecK7Nhv0GTWBlVmKEH6P2TKCJYvYxzyZRgTOoEsDoB1b7PDxkQVf4T+LDftHj7uWbPRSZDwIuEcNH1vseDCEhsfrDd/z2JethgBvePzQYkfrrU5hvtjmqK/6fLze5h2/9pfU6rjf6PCof/IQmt3uvfq3p+/Ovnn1wXn59j1MzjhAQgPKPHiThzc5Dw9f9oYHPE2BEA7ENV/GgXAA6Fykxt7enidmzJn5gUhNrzuiXqPEEwmtmUSUw0No70X2PJjiqqinwBeNJ4k/t2fT6/KJPQ1E6FVayGXxvxTQqMPin24GHMYz343saPqp8nXxeJkFlcdAdolI/Sg89OwU3zSAxr74wlM91lgAMegbnrg+LKm+hQk4fjiLbGQzIDv8fIYUM4FPYp4t7E+RH5qexWZdNosSNmN+qPDlz5huJK79NEvN9W+6BcivAOasBJEC3wjPRE638cfat8a+0e12qxOD/mYodvBzAAgSgO5mwIxeeuVnC9M07CCaG5buFkRBnmBjvexpvlzy5MbEKau15zBZwgAwsWnwT3maAZ1YbHUrOw+AqwkhxVBmxv7+qvikQ/zXsVjnTzvdkXUr+XF/n/3wPzVPshWXTeG+071l8TW+NCoARfo5F4HIIlYBDGuDrc3Ko0WOawmd9eGFvLEYrS8DQVQDWfkqVh91egCvi31XXqZinoMsTZ0sAmZX87hlaYWoZsbXUbLkI7bqLPMg8y9hsXkH16MEU3nRZSJIBesMYLWzqHNbG5bZ1vUMwVOvNLqPCm1vT0f1KXVSv9GtwrDjl92G2GVrN6px2VEV+HugIpGMagtAjzTiagj56Q//nX30BYj/dMRo+GzzxEqc1ibDTBLC3eaU8GExG5hLlZC2zKropTmz5wHNo/XjaVDMsf3jldm59D0BBNZhX7E0S8wWKLJFp9uFJh2AYQpgcPmSxQEPI8I2zKk20bXPiykbNV4jwtyFZPf31RpJ7q92ZpBEd0FkAnMrzi+42+ZxDGgwjdOyEVAeTAZFpJRICGlSg3KupFOtmyDyHBIAmzv55tuX8GQukmitk5oYB9xfICFfFkKla5D8vLDYJcpAOaSyx3Pbz8QyNbttA1tysAmcRAS+uBSbB/deNhgxbI8GWogUDLqSuWg3oemXwtOAa5I0ir6MS5GArhKpk/qhgzIHuothZLBShh7vhn5nxhnKqBFaIew3TPawRNuDpf48F4yjEaplpFFXd03hMulsGUnn/Jbp16x42lXTQPRe6bEqwgGhFqFCmJx3N47+p7/9u//9v/7AVle3CpDSPMb34fehWlz6pAua6P3b12TXGad5hjaTwfE3EMGbfAm8gk8WdAVsQcZj4nNCNbxwc+4leSw8+uADjLRAC74OwbDNAcStUniXvrgSibPIloHpjX8LdGuxhfDni2z8EE0qfK8eX/mJGH/NgQMtBmZfzNXzZTqnKzV5sNJfSW+AHb5kJprTAowH4N0QDXt/lvClAOZPgKkr1AIUGgNU8hzAdAFlgQYRmvyKelCTe2BY0MUdZoWh/BEb5wVWQrksCu2wIk89/xLE1U0gxh0549FK/r6Nr594fgpy6WY0C8T1Ew7cGPaId0auQCfoyTqFGWgY+LObnrI1dUuE0PMAd24GYmHkRkG+DJ/MeTwa9KGjKXcv5kkE4mr0YDAcHA4et8GekkHVQ+cjT0eDI/gQAEXJ6MEJP/E4fzKDTnsz8CeCm1F6AxbKspf7nWctoKrzpq9S/4sYHT2MrzvP/s9//m9//fQAGjzDVs9Q9piwvIh140M+Rc+Mk85mNyzOg5Sztg6mGrwaongsToToPHstQpHwBMkCnb+3pwD+lSbOpwfTZwaqB0MNgH5qrkOXKk+CwJ/CQidp4Vd9zsHJpCafgWFmRjYGBs9MdNls/GF22T45Lt3bgpCQpEuC+Ix258z4k8vxCl+UzZDcG82MPyEeGBRtiA3WGknmQOTR8Ey6lxNJE5fGeSD9SAe8ogP0NMarJgl3qiQMyutPV5/V2CokrJkpccedFfy87WjUX/letgAK6//myTpxS2oa9RuU1kZl6+TZAVEbRFdAPHkQpCD0RfiEXSe9FIbv86AHbrx7AVKm0wAuPys/ghWWw28QaZ0zUYn0iP9GCQ7/CVjn4Eb3sigeHSHJPuUMJMysmD84zCCVxx0wXHh40WkQqFGnzMfTx9PZ7EnJBoNDhHk6TfwE5VXMQ/JvMYABPiKIp5/+4m+eHvCSOvf2HrDez/jH5oop0DonQkkB4js/XUbSagReAw5BCQm6hEUz3wUko/4rXFkwL/NQmhygC3GcKYzd5Z4IuUfgTKkoe88KMxEuC7XQtdkLaEi87YlLeHgJJJWXUhiYHb5W4ZsHLIjYEhwLWGdUuiDiKIgB8LicgM1Q68DrMF366IMiJi/9y4iFPjRHxiVvF+Dbe//62+ek6rQ+cq64n5lpZrEQSANclySaQ8O0qlxigV0TWkBZQEdT/iliQSDIDsCx0jjYnyGIP2PLCMeKlia05pc8dEWhV64WwHzUlfYb02xi4D1YS4UDic88wK9xXvK6HpgJszf79smJRc1mYPVZqNXcMZhn8KSzRIMCPaqqUOph407W75yP7P4MDf/SkaV2aSBEDHCPC+sJoYskqY4h4T6Iwnliv0qSKAGB8DXyGHgoCBzaQs9EotKzjSMTybjEZLqIrhw9DydClxvWNvUxmsfRBqLFA1KTkT5ceO4tQQ2WallKYT+ENXEzGSm5wIBLE7Qy76iZDQZbiPaKMGHoz/Msi0LbDXz3oovCHXAPNJVWRd1qHd6ITXAygFoMMlxIx6CkIlwtadSo6T5g36n5obVGOohIksjlUnwZIXG4IsgDjhYG2CdIXx5alyFbiiXQLK8OqWo3oQFkvCKSU9zgARN8jJDi0uzHv0f6/+EfDpEItGWkwkc//PMIsJqyKahUZN8NdquSXacyMgPseBYFP/7jPMLQnWLFsLCHyxVXxhGFWSQWYlib9RBMdV5xS9Am7hIn6KcgXlFW4uNn7BjMccI9WoCqY7A+YofGYoLiDoGXUYtYLBAzuCYxDsyyEEL/ckjvwCvA86VwpnOLYJQhzCZaZhyMc4poQkshYBbI4I68nOe+h0xOXWlrFm7AFcwEWrTgS65BpKfgBADUlOKBCB4QknFwgHXsFEFK9wdIsLgGC5hfCOWjrgHWRDsGWj/TMqNbEWdENwOgAx5+4VWBdlNK48LXAUoCAz8EpUBrI6oGMuGa+SktBa0YIVc/2Sg2pGUHolzGggNl4QHfzxI0YZkZkW5YRJ/I4aMYTtq1S8euQH7ZB94BsckovI2/0C7DUO7+/uEA5N9Afi3COTQrw+Nmt3U6JWD5eIwf2mCi+5lDszTX6akr3TP6QI31k4+DUlsAdhZdiNBZiGvzobLOMvK6ZiBd+nYfo6noMp1JJzCUHhWsHzyUng/J4+KGNMSISQ+pQgZKnYDEOsdWfuWTrA/XFb1wS5+hTpzAUM9xrFnhdkaBh1I0APSb2KQ7GfWG/Yo6wId2HMUmtLSoCxAF+AKZEukQ2BGIteINKY2F8X9+bao7EAlBxDNoXImu4rulbFnCjEIHSGEuTKmrvQZknCp+4dWeKuWqXWO8rQzzKkouzAqcLLkZ1VgKQSSknSUNJHn4R0uZbdJWdVaVQlrskIchrrta9hQC50546NgTVZJkgh/dTfKpIpDuHuUWgXXnx0qIjeWaqzuKw5WiTkp4ZIk7wVFscowYItx12dMx+F+lHTfGi7vBSGE6nkYRqNdSugK6NM2N9UV3jTw03RWkoh4VDcW1K+KMvaJfaONwsJjrZKb3/sCZEUhXa52QKUa+3Cq7icHJ69qOg8TsOLdggYlbY+0TAkYfFWBtaeODqnbN+kTiRCKx+Kx8PQMLIwha+ELaqAD/Y5ILyVPFbqb9ka5M5R8hnwHHcuCFcIzNuzZgM8nUKEoz3JBGRcUSl7yq8FsiVrulILiselhJRZEMN+A3hoojYZc6jKR14AGZSO+kEVOnEVBWeewBf5gp+CxuJryxGlijIYaD18GR0OT7+8zcboWDK3QGaCdnoYxg/vTv/2tjA42WBni/262aPKq5xMBuen84KpU8qH61C4vREQWsUPCkm0jCo/yALiqOwd0q/qOyKcCyBdeQdt296AlTPXPthWqdXll+HcXdnQBa1x7AGI31HsoF+qjDxHeveNbetFzzOsh7rHojSnAWhb4gj7EIKd+wSmgfKaK29MSS91n4w1EJ26qCxo4SdwG+Sfr/fO0dnJfjrK19UhWukhosCoijxadIwsIrfKZaXJLpFcLKmOa2nWO17QtNcPUAlOnVN3mLnV4psOS+riHVpSE9FWPuJ9Hhw75RbAbT45pD1NUWEoL7go6Rh8FH+4sfS6RTAgA8Nidf4vMWr+hLrLajJucYtiwSBjbzAnTZRtUq0Nek6qoPa9Xv9hqq0qpOTaFCocBexkcAWc6myTY//cc/Z/v776TQQXQjhdwRqNjfpx0LnHFtr7wOG7SKn0Vq4oWUnPLMXZhyKNI5U8bOVp5RsX/67A4n5pKDm56S/6IjKhscDFhmGEMQ4Uu0yyfnhZ3tW5ISRZgvgTFgrajrbkvsx2cHYG+GqkER9/k6on16wNHg9mBVNritRHmQlNB8oE241A+BmMGENGfo+iaKsmY2GhSbbeGkYgMjPELquM3IJDyP6WedmRAP2hQHhkXqP6+3ABxVtrGAYopQ9ZQyt4BM8K57y376i//Q8jKZdABq57w2+x0ssHq//+mvNvYrra0a38nVre2tRXNNhcsbh/RsapZC+TUPBJAOurdSB6dg1fPEw60/5GIMiMWJuIQmUcM97pY+OG4QET0VtOTdnckis8bsb7/7iNlKKq0FvRQMwElDqUJ6ihk0BD+VErKOOhTyfpiXlOMv5zsJX5SjMFVHTtWOw7lBAgWfkZNFj7qt5vtOyT41qau3oJfzhu2NeNRrb8Jri60tvQfe7+D43AbfPADz2DRQSxmsyAhShECgJqOH/XO1+BEAlctvistshALkA0nilzzjCo2AUlpG75dcwr0NuNIrSBqKhzfm3Ugs1utON0r/23FdVVIUKmrEwQQwZPuhJ67P91po7957rzrlcpFnflDGBOjWdqP4pg5L38BQaMkdUrGO013rA0MPcoTSBlnJOEpNXqpXmMVmg6sLFgZQwvrg6+lumpqa4qqiCnia/v9p3/y6JspuZkkN2n0NLatpgWywOhZo2EAjRYlfkLfvy8+Etnae1ioHxMxxIWbAOojyxBWFjqnTomSEOK0xgg45RGD+Fw8vlzBa2dC+9JMs54FD2xE3lXgBCnf6zHZzj9vw3kGDB7FiUiin8tJPHX6J2cTTAO07QizGZfsV5dxmMNV3nlcKRWjwJP5UmMKOu7eob9+fvgOFfLm081R4YB5hvLc/sgdgOx607G8AKGgrE7iKxmhovn6OmU1LOxYJJlLc/oYyxF6ffcfylHt81A7LnE8G52C0zid9IJZG7yt62eylls5Wgip2P1MKunN3IWBeaMsJm+4Ke6bVlrleSznBzDhxjUTpvHn74SNFmjMdFZ7Hub68TPhSX1cu3bIF7qHSdbHdJSOAmtbWyQsJ5BKJBOVAfAU/+nZ/nS5RQJYJ8WRWGiHwrc976dJnvR44rMlNDwY7RtD+F47zteHekjRJi66vaVGt7VtaOBW0sfNEEJg4Allkewm/gt5kbGzsppdWGC0Ex+htGOXA46mxNYSYLkQQqBCTy2OE7kR5FueZeoiBBH0Jfg68Gx9iAMyDqwIw5v1OZFj0Wgrwa0pthnaJH5tduf1ggpVpdIHc9K1ldMtII7o40EdsSaxzaEeJ9GwfrokYMfDOJ0N8DOKFTw7P8edRCQLpwcmuMxlrVF8N+sMjIuuDFX5cPJE0bdyhtCogDdzfBK6Sn9wpbS55kJJ14GYmkgHOD4CNL3EHDn4V/AqIh9cKFFw66jmqBlz0cRaXCXUXjiVxi+B1jmI5XOKXyYVT7BlclyIJ08WK95NR76RfC0YhQKuYL1wtcSW0I1BTF6UvkPAZHlDRWQvmSZ9iRO+jTxHtvnkR48EcD7SkMl8t8HFXHNhnwW/A+A7KqF2NvdoHA+gs+HdvB99Ip6zX0mRqyWCdZ+/kzDDkg4lscAfSnvwklbiyzTX+FE1T+ZxnKLNCWGQDH2LOu8HdDGNTuEWGdljo4HzxTmXDf/P2XZVATh4pCjkZShI56UvSODlWhPD4UG2+gQMm8+hNAwlSSUX49eB4ejyd9fHyN2jsmcbvQMsYWlZii5Ppo+lsplvsNa1f1Vy3HsxOvIfTCrwXsj9X9Sf6vH/Y3wjtI8pfLYex/dFseuid4OUP//RCs//vLfYGZkNZlCdS2KKFARTgLsA9rccd0P3M6q6j3FND1gAMWywn45FwVNICRhAK6i/F+CWdVVmQUbiY9EBiPBvjwtRaYSIaTPX46PGRZ5DZhZ+RPQAdVh00fFH30Gi49R28SntUkgtMDhjUv4oznLWhc5lBmv0e5CDGUB6fkDCzVm9Ad7+Bh5hUc02SskuyTerwrbpEhW6u66EbmP5Jf3TeMLFpIcrowtM4Cm4w+5bFMLAsHXdWMNbbDsacgnEnBLsWE+uS6AJYbYVZZltGMjNkyx6l4EH7oX1cR+/AfnjbOXhW8aOJIirjSWMeataOuYf7N6PD+JqdFMl7RZLesJ6ktzHzedVRa3142KkPp/NgyIfe4cPOrcophRnqFMEBdJpGAfg2mwG7m8B1nq2QNm7Z0+mzFdEhKahVfovJnk8PcJYaCyhF0D0DMSO37bVkKSxT6IQaPWP9kqxqhEhmacBvyWCLAp15/qsidx2xNMgNiNVN1xqqFNnbtoxd6EOF8FPCZIGjjpLF8rQP4hSNWQKL7QgX65gGZeUQraM98ZSurgfjTr/DrodArb8H6r6B+xWyYd8e4O2wdqsZQc3l+Fg/6nk8XfAk4TfjzhE7QhKnHtF0w0B6S2awxMazn/7279gHSmH3ouaRB1pilRhMKcKAOikuvy89FsRifWVVD1LUQw9/8weZkSWPK2iU7G3OCa9mvg77w+Fwus4U9QWVo2oufTO7tYXENBU+BsD0ABNSKReVFbncTxrJ1NuSZmvJ6820dJi4K3pTkV1h5i5lp18lPB7hD0pLp0zwFS7a7VonLRjWNgfqZHD8lHVDKY/ySdMVBBFw0AS82QnEzMn4St7+Xi1b1YipDu5yTruNz6NroGfWZ0DMbPUGSBYPQ4rkUpxSAuJ7dFtKsb4hX/oRZuc3+5DZx4gi1gckaV66XXU6klpJsyAzwmB2XyOF9sZqQAcaLMmw7m0jNX5v78UHzJ5Fa3MaeTcWs2VeOS02h4ElbFXL4H40eDzw2L+SHiIHt3qv5QtwSXoKH8d9wEKl/RNWI8PbvQcLL2GLAVuVOdRDpGCmMNVXTWK2qh9Y0A2GiEqGzVLfE43hTgfe8OgJW+M7xV36TcFMiEOmuQmPVyBYTBL5NeDicO1pEGHSkepF3tY7k7KjhkMFXSFE9llblAdz3IerQEEi40mP1gq9qJO+J+aWMoytB4/cI28267Yt1Gw2qz7e08ZSkfH+qLHA6/Kp8fq6B1Lei65GyGLDI3ivxnF8DCNP/HkKLpGNBMxWirBvmX4e8Cn8XNU7GdaJ7HZvFkUZUqJmEmTVWgOkeLWnmPuB5+S+duA2ODVnaqssVlkxYklRrsSmy9R+zlNhxomP4URnkYux4YeeP8eUv1DkWQKOMD1NA7Aqt0cfELfjSQn6dRTNA/E1PDVlBQX0OYxCsKOmS3mY9tC4x/RxOwXdXnSAfO2UlOCQSWooToZv2947Hk8uKo1KYEieLdAkjSK0tgYanG7VBEdr6RC9YSsi5xJW5W0BSDWpTxKDMgUUKV/0/MpXGoR+3xgLkVcdkpQ3xXgyPwtEe1+VvYI4z7YhvbXBZqxTiruTCpCxHtJXywK4w9khDbLZdvtI7wJdYHwH+BtR6y6EewF8r7DbMnq95Ntbblz+xmfthLCxUWPckt+ngpJmKgUq0EI7m1DIwJlPDfKJDx8r01EWu5BHQvxsgYLhOZJLahK9jI3TM0Z5EnhwALf3YUDE3mP6CaNL0zEo41bB8CkdG2Z3/GzlRS74pWFmI1HbbsDT9Bs/Bf/F88wOzgIPkXcx3uOJZeUkrB7S++jKFIFYOr43NkCrGo19X2jyDqCAbEbv+gFrDvr7cC1uf1bblq919Bmjf+osKCU1N3q7/8EnPE8Fo/G/fM59kXhRDZzu+wUdkDRTF8zp8aGFoQBHutKHw77Fivmj/m0ioArnI5/Woogb2piGOpZobGhbTTqHT95imRCTaHAsEz2KTACkiZsYiCX2A8PaeXOWUeUR2pQeG+9fPz81iiO4w8N+dyMcPYlT1wUZ60ehabz78e9TlZ/PTAG2cuBxLP+QqJwbTOHvAnjc+2xd0019IEHc0ZS4Dhi+BU2v5Ejq+Nky7cFwy7SL/BAxa1uSM5os0pj9y/ZHGeWbOwQl9wt2KE9urHf2pnYGo9HfvYA7uqQNdPEyiWISGnciYSKPU1iUZE/59oQWtGHkC5l3T2+wzS5soL7Eb0o4RQf4iG6a8I3zHdLIeYBG26YxK6R+C5gIVUkB5BDjXjxXnGBm5nt/XnBXI29nA0VJ/L9HD8ykgx2YvIW753Lgp/LcP51+k4NdO11y579CYEoL/H6TOw0okTYRPLjHxCQhyrl9AL9KJGbfHlrs2GID+5E83DXu2/1BsQRLkSVRer+xvcPDsrhsumbFPeRa5WwDjfKFMi5MuSmoRqVqXdD0KQVsQ22LO3ppIAJmPcTTRX37YYGKEhPoIYSg27h3P2y8gCHiR/dAAkVFsjUSVIXFCipEU6oY3hnggiogGfciQpWAMy7tsZGsNUa57vJs2/FgyG6wlhfr9/uyngkzEzBUwWTCRLy1NO0d/0njjg54LgFkQpXIbCbriI3YsSqGZukGAejy6I6VhfVviM6JgTuwKK2wWBn+xmpkmActsYjb/wqzE8To+eTwvHtfHKpF0PntZnzdvWOg91TgxdmobWzxb3I/4wn6ud5deFK4wh34bRD1hGRZiPsR/u9k8SEzijGBngdgMaIVRKt4L1uncs6oMVZV9EQNVpexUE3fnR3dmxfqPPFb3Dh2wYrEcn+jyjkQH6MFMsjPbqhAomBZ5CGN8pTVChbt3XmaqkGvVIWwUDRlJUI1SVXo6Mf/EdION0G4iyf06bNtK02rxTALjGEC22aI00yJZnlAvDDT26uHGBXHYC5C1Jn+F5hXMN/SBfHw9kywaiFBygOTeZLdyejx+Ua4WBjkejutYd66rDuYmniImQIK6XhClsp5ga1XnwS02agYa27MOz+V3l6UbnNl5s4aZk9dTPTwv9D6S7ylyy14m/NAAngNclskN9o4/Ri5WEUCEyOwQAMW0QjoREFZIPQ+nCIr5aTjYWHZHqMDSGVEZCaruFKMvRHoAvN+5GC/BheI8vAdN8rDDGwOLEgHK1AS/b97e1bJ+94F6aeXeGobxeBo7xcQwXRmsmYuHNImuMXqqgOziMhyGPyRGuSMyn+aP/3lfxn08bwglZm7g7/lKdja6AaY1fHwaG18Az2+w2FpZdPnW/qg07V18Lqsp/xd72Oo+8Din/dBgxrPCzIwQFegGbCL6bnL+slT79D4t/lyijMYHh4VGPgglr4+SylcKo0yvsNVK07Wb9VIp4HgGVao2Ma06pByHcFYOxXscjRHlSV6XED9GpqQTSmLZOCphy3w5fHlGvTe4DHApR8aPmb/aDXqozRPfbb0UVpkCWpt84d/6v4yS1Gcn27aaUc4nkcoSQZ9Gt/RUb+w0jBAqD/cxcHcQl/vRQoCTKrQXdSnOsVdw+CRKpULw1KvDU31JVkpy5/JYpT3M56+ibC+Q/XY1L1MpqmjT8ndIWDpMcYqQMcZFBcxSi33deW41uaedvaWi6CTtg3ugvyzzKM7oa+p2zNKrwXaDwD5d34uj6hVI7t3fXFfjbeNqddH/zuRsDylqCdu7x8wmau3g9mQlCPbMJc/IqIsUlfAyvg7xJIfVYPHuF3bprSxnHKeNka5ofbd/r6ySff3bQZ4ARbCHUSR0OHxHX1VeYwdFiQX8+qhczwHHMij/hWPQJed3tWOwuC5Mp4etaiaeqwcNx7AgE93iJkPypP6YLyWh/E3i4ulg8f/JWLffHz3jblWQen0c/7jPzIOxqNwBWiDSkWjHSyxYfUsuRxTEaTZNqrCk99xUK2n9HcaYVsBKDXS4nzN5pHiyDYMtKVz4LQ10axY/iWwDE/mGFwB7pezwMSor4pz5ywFwSlVwU5U1ipiNhNbi++uC4+Rr8tBv7Ef/qHuZv/wzzspph0NhEuq3y5xQ2Mw1+V6uQnzeIuRdumAG9sC6LX2cLeDaQtrqtxtrLt29h0IWJm8fFfsdalDmUQe9dT2iuBbRps0FrzZ5hrqbPl2Wf8pmirDBa1A0zBUSR9d1ofHcXDjyICjTBhuVC+BCYKVYDFUENrOx3Z1L1sfpdGtMaiWVbqSHdjugodzYVY7tZj+PWkv6gOc3CyxgvqK8rSR2SwSjZaSY5aUHJaiJEvSgVXynaU0CR1KKydhAoZlrTuzrJLWymWTX6l02q9UP+0XLaLWbnhPgMhK/MvV0EjGkZ9jpnzo5gkYwu5NGQraa40tlA3p9AZa/lgTRyRkYu/vU71ECb4hX+00d/F8lFkt+mIxGt1EkUV1dPcZF7tHz0Sjqt+SXn9OZ2VvShgoUq1KE0vVLoNnlQOIlcYBX07xfJ5ZOWdvlYcwuxrAZI6jpwBNhfHgoS3PnZrlUW39STnL8oNppWuqLwGQp5oHyWuwtIl/Tq/gc0va1RsWpnLesQK7ONGpB0PmbPUg0jKSB4v/eAQQDPBacmGC0OSgSp27CHXJrx2Sx3rTV4lIBLX386vTUlHCZM/58Or97169r1dqnYF4cTAjz8Qf5fmpF75IsPiOLFxLtURdEXicgKHlk1GVm2meUrgQjGmRZFh/ZJqsVRZNI/dClCXwLmSRAj/EfEjcY7kS06rRhGESPVaqhHfRUm/Aw5I5eHC6XjantcIGAQUKiFLR8FU2n65bOxqeNUdlZHkYCtxRLQeHp3TbB5bZqqpopseAqIgRFQnpOcSUxQhfX7GjfreROCRxaMtf0spPd5h2ak99rL9g9G36D8NWLZamrhvahpxvP1BxlnXYtQIVGgKMX5FWwPPQXThymWl640ePH+qJ0UTHa/RXcBC8K5NCS76SQE3KA0+oUtu4MjX1mPqS+JS9q1OieK5BbNp8x1i0CDNH1lpzMG+s+Cy6cgTiQD3QKRqDtpAlRbaF51AO+XjSPIVe3YtQAkNT1USzBJVbRNZfFwXEs27x17S8ImH2WtT+RFKlibGl7AN8Vi33WD81DCJqLigrpPeZfn6LGzTXWxIXjEWWxeno4GAOJJtPbTdaVkZSHdQBWAyCpyI9wFTcNDtAFx0F78Fmj9uofN8DROXXPb70Hh6hBKW8wkaNDpiruwDZjbO0WD96dHxck67wuE6qIIfwr26lIJlBi8BkcyCoscHxfKCm30G9npFuXq8lgU/WSpTmSZDymTBPuv+SDFClecbzbDE2cVZWMc6W7Wts5iyBDLBCY1vS5O688rPYQypGckyltzI8Xk6NJtOg9qhxDCAabqmU8ZpclHxS56BqRSi3fnj+jOqLTIiASnlv9Hph1MM/5yFrbMhH0O0urvaMmAR4ZDB8ROs2GK1wqW53iVTLw+3j6gjfnr1CGxX8h6T6/MPHl4DUyiH5SmlyrCFQVgcr9QyVUa/XDcvYU/aw36jpJM+dYU/qtL2NZBDUy0yQAYql5ASYghiUMJNCPEx470u/d9I7/+p7G5anXI3vUWRgYBO7wL9XYayV3FmuKyK54Esbs5Rjs8VBB5uEl8U/N1XT1CbfhEovo3p2UD1X5nm+pdhmnSgVrZwrPO1yJFzWCjXKP5XYOPiNwrewMnDKazUDFYTvUl0nX4RYVonJP9YYVYrA059yTGRuaWTbttEwgZu2UrtAWhNKJGukIGgXNZslRyvp31+XluRAA6MRoeRtWaI1uaHUArSWkr8c+N7/BbadQD8=", "estudio.html": "eNq1W91u20iWvvdTMEp3RCYU9Wc5tmQqmzjuJANnkonT0wMYRlIiSxJjkqUhKVmyImCeYG8W2N27RV/sAnuxwAJ7MXd7MXmTfoJ9hP1OFf/0Y3e6e7eBtMiqU6dOnf9zWD6+5wonWUy4Nk4Cv793TD+az8KRXeFxpX885sztHwc8YZozZlHME7syTYa1wwqA5XDIAm5XZh6/nogoqWiOCBMeAuzac5Ox7fKZ5/CafDG90Es85tdih/ncbhKOxEt83j+Nk6nrCa39/LiuRvaO42RBv91IiGRZqw1G3fvNVrPdPOzVahMWch/vg6bb2se774W8e7/FWm77AK/JPOne54f8iHO8BVO8HbEjlzG8Mcfp3u8MOoNhA2/iCliGR+7BYLX3cDkQ81rs3XjhqDsQkcujGkZWxBNzINzFMmDRyAu7jd6Ye6Nx0m02Gt/2Bsy5GkViGrrdGYt0otToOcIXUfoOaozeEGzpNvcncy1exAkPalPPrLHJxOc1NWCe85Hg2vevzJiFcS3mkTfsiRmPhr647o491+Xhau9+IibLiYjBRxF22SAW/jThPYyCmMm85/Nhop6ilEI8ul488dmiO/T5vDdik+4hBpnvjcKah53jrgOB8ah3U/NCl8+7nR5B1q4jgNL/VntWzEfLNTRbp5YiMXqKb90mHVT4nqupWRJQNlmLmOtNY0XbhLku8bs9UaS1JnO1nTaYJokITWuQhMvSbkkE9kxYBJLXuAwp57s3NnZ6XNoIz1oTuyiJeOEYjAamaRQD1UR4khPXYzCmhm0c3g1FwYOUKEuskXS/5bSGbbYl9LU1XbCPDXzuLgXQesmia7U72b4uH7Kpn9CCyZL4223i2XULUUfcZ4k343K4v8mUrxfBDhpdV7MCHk5zAYci5L1tJVMa1VDK1tyl+b9RBw7wHHihchbd5hHNBWxeS63toDEbFxbBpomQxFtiwsONEwx84VxhlgY1ZqpfJYd1mF66FxlywudJTZpFl8xo24YlgMsdETHJGcWnlPhD0iuieP2AdKYSn+SSXEvvUMGc+O6Yjrx2BDW0QwOzRXHAfH+5ZR3gG0u6UozwJHw+uUWPxFUupjIOGiaKa9eZPBpAMxAgKdjhk9REV9qadEsd4jFZ71BEgbJj6DT/k17DjFHyPmuORroi14u4I9GDoGkQ7vJemWMjjVFSPTqYXYNCFnpBvNxygiSZdT/X+zSNE2+4qKUxLMWcocgU6DfbXclVZWrS2rCDn3VRG0RZu+hCqNspSDmuBu4Ph0Og8sKh2CFCFU0QtKTNHxCN20pFGoGgyZWkMynu9+jcmek2rQ528fmIh+7yNs8i16da83jD9vc7m7Zftqo0EXCcX+16cuvdOM6aS8zPlp9Fc73Zbs3aoaBp9gC0WmOVIfCWmQfCojyt2HIj7RItXihZm/q4+zDqXTz1QiRr8DFr1O2garfOb8s5O3x7teewcMbiTWd7XE8ztuPYibxJolFeaVe8gPLCgE0q/WX6Ele6y0oyjjivdCvjJJnE3XrdcUPrU+xy35tFVsiTejgJ6hLo7xpW86BhNeqDqee7aswKhDv1OVZUTIWqDkmKMK7/MpR8zgLkYHH9UxzUK6sVDiGJxynqKvelxA9vELTmuXYFhoC0VdPkgOOzOLYrCPIVOUlZcCynAaAMU3NZwmozLOTzSrZAhJX+ewSTacSO6wquvwmPFAeM6r/1+A2Lc6idqJU/JmAei/hWjNh8Uek/jRzP91kZ43Edh9k6FM65c8J11WF92rSS75LOIqBU+j/981+0s+mN9tM//ndBTQkFBaldbPKp6pCVQCUrCXafhgDnUFNPRIBMnzQdqm/cwSqswg/MyuFY9jR7vHULN2IBIr3DKv3nePzyIz3fCg1bCAH5ln40PULdE7NIS5SYC7IUT+/krMOCYDdj/+df/v5ftZMvP8KVsF/MXceuNMxGpf8d5c589zkA0zyUUKfIfn2X3Qp2JKHOmCs07+bP1h1sB3CtBA2vZt2Ktd0xm6gt2/X9W0Ea5mGD1DjyBrdTN0TmXumfsLKB/TzjkRLt5vvpnPwWxHknzzMcSgBr+2U/mRshpy3pG7EQPIE347746S//tgkXJ2zEB8wd8Yom3SuIvTtLp0jdOSwKwHJQ7qwFzbQILhKB7QRkR5JSCpBtCYGSeQy5XncbWkNrUiBNMR8e5h6kOLeMetvjlIBsj6pkoPC3NCaznltcU+6IEyH8TUdM45FI1pzwUyQSNQyybScgDTGBnziBtUQlgC2k10hOYffC//IfIxHucMBSkFfcJ6P685T7PBG7YcYgDzb+D3/VTthkLToUXjr7WQuyKhriwCrAag81FmvvX747PdWGkQi0qox61V42v3wTDbzkBKKMwKhVGSiLo046WV8DRTwtYXlx9v67M2gy0tFdKHw5FdcLsPXlz989PXnzFetLcOsI3gkRnIYzLxIhbC7ZiYQX83F9Y4HCtoejxon2jR3bfVc4UzkDOUWLc8jKSUSkx0Yvhfru1dnpub2EW+9eVJXZfrjmA2vkD6pmNkA6CpNYH1SRwLvJJi5NGeKBZiJjfDqYBnMapmiejlLsxhBTwVsNrjKafv/q5NReblLTrb44e6YhSvNIaBjUdAQxR2iPtN+9PX1h7KZWLUpHNB3LTO2FcEViau9l3vUpLq2Ei7GGgzlWfffsT9T503im4Jr+feglC1P7Pow4803ttYdUS2DxnpYutwY+fAEWP6NfjvitRozbeabIi73SRgUwUgeK1sISg0+AfPPsd4BKZNQWpW0/QHsiHsNptl0rTnyAnr8/09LRL/8Vau3nmh4EJTJ84TY/dBopCWdvnmdsReG6BtX60OyUoILpIoNsdr6VNJRk3a2q5E6DMmvjKaQNXIXQu9XXbMIQLjU5hrmy9EE0/LZGpTTD1MxzufiAxMELWWQFk30A/JHGlCwdOlcONoL+t5H+lsHw/rf/lBDg4ocJpS/MmoQjALwKEINCIkROamqyusotJ5Li45Ed8mvldawf+ODF2bt0XF+ykPq+HotR9k+5SZzm0YxDIa8Rb55Nh0MUbTS1gqFl6CwUMG+9OfffUbtFf82SsYXz6aqlXMyYLcPolRedIzrpXhjy6Ie07YzHl9KuyujFNJlM4dkQ/s6p02cr0s/fvXhWDBaI4YY5JDIBwSnk05PT8+88P/Cc98VciRIZFjFs8VB2/mw6YW8v9zFUWlhYhQUnY5Q2er7UhZvyOQHljidGYcZLHD6nd93oyXGriOslEHkKvTFXbfMc0ySIeFACe/v63enrF8ASMfJ1GRHUIiBwOFDUKrZcZZGDVTvT+g13qhsmaqp9w1Jmy7MNoZbl7eDSJ9TNmfETFmBTHUlfIa16SVqErmm2Go2cdifxJaq1oKRjA3Mn7wCeMv856jySnBQBDVMP4R0if8J3jZ1POOTVtFrZxqPIK7P2BV5fcn8C5W42zH1kt/MWbzfVQ6u133IyySAM6bQ4P8LQF6JsKq95PNZL7EH9wF9wEfAkWuitBo5vmCWxp0pFJQ/z9XIzeWXIFp+ILMpoKCu05nZNms3bV/VWNoeSB6xXeNTRC0IlSE7pmAdeidCXeI0nYzD5jIQDxRrK/3DkNv7b3zetg/KpaXmhvtOwhOp51s9j/gauFjBMQ8thcbJGIsaUPVkBm5CBk6GDPfuHJv2vvC9g820jL7h728NDxrBts4wAi4BgD9FFc6YQ1dT3zQDuJn1kEkVsL1cIy8K5KlscvcMMkBWXRp+JeRuDL6FPhyYlgkpzFDbKfAkVmzFP6fY5DpYfQOVAcrxIo/TMNl2K6HKylCNhVo4Th55zB8Epegsl0Ku/ti/ie4O4LlHWR34yrFeNniJL7lDaWMIQ64bTUDJJk00C/cpYIoFVJJ/Zy7TG714UUjcvWua+2b40C4lAm8ym1TBJ/pdmVu6ni1qua7YtLOuYUABaNhi6h1hmdUzriJaqZZSs5zW/WssaBzAqQFzU9rGxXHzQeNygPYERhiT/qfV5C6BEbBt7XLTNjllr0trhsNN5/Ngk2Vogutkxh8yPsVY2BErrrH3zooFlHbVKmQ7WHch91aLVxdVlD0RLGyD3TSx+yef62UXjUpkGtX9DVH0L++yieSmHsipQ2oRlWWcXrUvDCqZ+4k38xbnDfBbpLxEkvWATa/tSDZex7ksSaHQNca318KVJ/2rth0BG5r22rHOZ2lAp5cbw48snKoR0Sd+3QV6VUBzInb2hfo/WGcsdezyyDlZ7q0LDBrA13VjCUK3pBOU/4nASefMfRISASkKkvu8cXuOKnwaTZJHbzmDLQmlvwoNyb4YgxXVh95cgRlheTF7ayF7OryhIuWoMdnYlM9EwJQCIBNgcILngzygkU4pDRPYGGJ4sgGJQDBuUAfgLRfU+5oKCfmMli00OzcA2ozQo/AzuHOyrNyH+TEMwUx8YqxXlUoje4KyYl/g8hCVwnd3YsA3u282WtGnF/ZSjjk2YRjw5kT3kUlD7oyyjwGLDjDMgmaXtAum9tFWux+Z6bC1M5ABKNMrllSfnJgHE1o3xsGkd1eVMwkJKCKyhmD3Mgh+yWwMRMEfCbHaTT1LDi9vcLw9IULZhAI41fyRhUIDozFDwjoh1bjx0TcdaFLPpyM2jHGYTXh0JSQe8E7ihpEffZbBryFlku3UkPvJ1SG8PO+pF6djbSHxSoUzJVFe5Tq6AZQuhsiiGiVA8S+yGObMbvbvU/MGDe8KaIkF/Ds9pUXcD5qWSILtQr97skT2yWILdB1DGuGCVA71LeglNy7bTk/RXTdTb3TuW1dukgLmcvkJZAPuNXpVfsKqGpZLH96/P7I/fLCW/ZVKsJ0hKxZmgeyfn2Doc6cZKw8OXH8PR1Bex9re/at8sZzuAZl/+PUIA4PHxIOp/s4TOAeg7ZAOu3sL0l3/SaHCxa/BmbTD4+IisWSdKLVnuxU8+qn1LQytVCcYfu9UqzJ/Fi9DRclHyOSqwWB8aS0ggFUpks2vmIa/kiTPWh+YS4hkLt1t9efr0ebUwZxQ8VyuHERAEmg7KqLNa7W1uFI/FtU6fM0qR22GhG9uyAXJBU5c90qih53OZyfRQi+ppiquJoQI3oFcys7HGjOg2lhJ+2BtEnF2tUmdPY4aiKJMnJRiQp+x5WmnL067SR6xqoR6j9OhpNkI/T+kcOiGETUu15pBCwnyjhJXqk5P0ntLH7SastqY6zUbjIZe4uVvPkK2+/aj0FBvAloylCmsokcSMy5EVZopcbwOgNNHbSAhXKgjBeOSK3t0BSWxlymJXhi9QuE2RdQUn2AGFjRR7TwYXoCNh2rZdpSZT1ZDBQVYXG0XKOTyri1SqKD5U6xi519HAcQ5M8Gs0Dnkcd61Ox6TbYb56a3QUs9Z2SxtdVTrEV+8INkB4srBNewmlTR+vEL7UTkUyT7LobUaqIJSOJfBCayE5nHuhRc0OwrRUKgbhMamGWx8B0qIq6BVVQY80Tbap1xxStZrqC9wf9WQkuOXzcJRAkCmm/PhPM5DXNJGfItN82Ze3L6rvOEiivtGJav3Qk4hQBuPhnPlTl6VPiXx4BkuUD29FzD+8r14qnGsUxSJKdJ2ZA8Pu63If5b7fwJItuu1nPDo6Mr49OqptzA7WZlNyC6fgkFMo72QsU55dOHLlpS2ZYDm+N3kqZygeSizZsQd23jxx4EASnlb6elX1yauUB5WtW2GGScB/OFyvfsDZNQklQuyD4k3HMcm56ArUWJNeuTkzSLMxCZxRngng8kn21C0fEUl7ilUdLY3P1NaQ2MiBwjGewwWkNnmbuZdTzmxF2jAjgsmJgN7sjpl9L4UpT8rvHmcIIJkXqkqGkbXw5AeEeZ2AKd6vAdOHAOaFsQLPA676krPlojet+/PnbER2u6tPqvK6QLVbpe9X1S1shb1sYnpSPR7kX7j7FI1PmMs06YM0W5uGKnJaNPNcQCdxLC3gNzhJaGo+0+hSFKNBVwx8ZlW7Upyb5NEm2Td3uUv1kQrZqnP7+TPi8qOqpt5MbMtK/duuMnKVK6NKltUvRHqmeo9pJta7PcDlTNn9IeKp72OpvGagZRqPRDc6ZYjqA7s/KMktEaORr4RsDiz6MAoxWzOcVcZ1mSOmLQ5yNsqLFVmjVPMwy/yQMKcaH172KGSzLF6rAEgYHjxIH+7ZNjPSZySvLn8zRVnaNnrMosZvohty9FVIg5bcyOhlZLCfO3352tPXnr7sEmw73PIHhlHOl1NrEKEs+O6R700Pq5hxHdsXl7dY6nrefI38dhqPdWHATvGSEXtt968tdYk1M8Zr49b6814W7z9/3tygSJpkaNkqTvWnUcQWGJW/ehFpjSfFc/eieL4sOBrY/UDuItXZBjvSVHKVb5frx/VmQ7MoFMz1mWcs9pztFOJwcDgYDs18OxXcS5d91UCp3Wm6fJKMf4i8hHdVGmvk0eJ6nU0qCbq2VI/4jYydLSRFlB9cU8m7yrt90lLtatqiqpa69duqmN3E3dTCLLRQAsph4WKCcm3CRjImFL07e5CqQBrE0nFhu7u9L0KRctd3k1SiZm7357vcvsRkSDsWRnk3Ykg6TXlUvhHGT2d4ICD6UKBX5QmrJkXP/yNyjDt5faGu2VzexmsiZJnLr/B3/qYTXn3NPs7P7LP+/UBFbnLqkUjuDLQeZUkZaQ6FHbqoghT4F/dRUtDE3p50rLmpWl5zpLMvH1qNx9SKyPLHza5GAvDEWuDfzSMAdzqqlVBuSyQb3YXCDSgqLpiJ7K10MgvhzENWZoIbAZvov58GA6p10iaSyX9ODHfGuLIsZKFaCm+kRoUk7pLavfX3nfIrR5F1cGOl9nGSaGOfLPjvlbKpdTqUmYf2va/It3o7Ycp0kV8uhayUrjTnu2Pjraxwx8ZbMJsb7+VxoFzGFl+r0s5o+qlMVjEFYB50LOnK38Pppra0C4Yuvaq/KWqVP/aUKmml33fWy78xHi4DqxSSVFgJrDQo2fIbnIwmac/2a2v+3v830RmFzd76AdK4STRnejPeNpz806p60OWxTDiSzAuxW8syVqWsb4ygbu/4QAt1ohj9/bszverRLYM63TigFa64DilDtquOugkl7yJgQpKlk5ZvxyS6v3HDVVBakp9j8kOzvfvj8t3d1K+9T7B5YSGv28+EmOgls3NRjFLlQ379OUc9rqdF4YMHa7Whu+1sb+M+7b2ny86d3AhqR21B+UVRNcb0cmuwGnmjD7KU+RTLUk7+gpd5U7Dw6CiV8m+RbwbEHGvGfNQ4qhZikTP2ZgLV0DLNujaaf7/xbpRZvidTvhSzfgnm0gCd6sP1LiJSwq/4ItbpjpSxAa6YA+EHXgxvgrhzYVn0eymDVtoSVdaoYPP+q6GamgpXyoJfG86Kol31VkvR7NKKkUTroGGtiVrqkAc2WW166bT6dVzY6sn+Aiselo2TXoqCnZBfDC8ffdSO5R8g9b9ZDrNEwMKhJzAIg27Yy8mPcEXl9gqTbnODdkkm0XsvRf7baC2XgcOd+yvsf5C6D7d0zknT3zKE9FiH9bI0a6JRyWuqGmb2H8io9eqsanz+XE2oRU0kp53y22Q4s8uSkcBQDC909Su7r9Ze3bYW28hmbRp/0/1VmkBZZbNq7MwbVH6fJqPp9Z4kI58nbELco555kjF6IPWruIa81WHjCQY2uhiqxbPKTUx9TaCOg0GJUf5HFsd1+dcVx3X118f/C6Rw9/c="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()